# Caso F · 02 Tracking de experimentos con MLflow local

> _Tutorial · Caso de uso: **F — MLOps** · Capa Medallion: **transversal** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Ejecutar un run completo del baseline del Caso B con MLflow local (SQLite) y demostrar la trazabilidad.


## 2. Qué se aprena

- `mlflow.start_run()`.
- `mlflow.log_param`, `mlflow.log_metric`, `mlflow.log_artifact`.
- `mlflow.set_tag("lakefs_tag", "...")`.


## 3. Contexto del caso de uso

Sin servidor MLflow externo: usamos backend `sqlite:///mlflow.db` y almacenamiento local.


## 4. Relación con CENTINELA+

Producción cambiará la URL del tracking server; el resto del código no.


## 5. Relación con Medallion

Transversal.


## 6. Datos de entrada

Mock BDG2.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica.


## 9. Carga de datos o mock

Cargamos features Caso B (regenerar si falta).


In [2]:
df, _ = mocks.make_bdg2_education_subset()
df = df[df.building_id == df.building_id.unique()[0]].set_index("timestamp")
X = pd.DataFrame({
    "y": df["power_kw"],
    "t_outdoor": df["t_outdoor"],
    "lag_24h": df["power_kw"].shift(24),
}).dropna()
y = X.pop("y")
n = len(X); i = int(n * 0.8)
X_tr, X_te = X.iloc[:i], X.iloc[i:]
y_tr, y_te = y.iloc[:i], y.iloc[i:]
print(len(X_tr), len(X_te))


6892 1724


## 10. Exploración paso a paso

Comprobamos si MLflow está disponible.


In [3]:
import os

try:
    import mlflow
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False
print("mlflow disponible:", HAS_MLFLOW)


mlflow disponible: False


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Run MLflow.


In [4]:
import math, json

from sklearn.ensemble import RandomForestRegressor

if HAS_MLFLOW:
    mlflow_dir = ROOT / "output" / "mlruns"
    mlflow_dir.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(mlflow_dir.as_uri())
    mlflow.set_experiment("case_B_baseline_2026")

    with mlflow.start_run(run_name="rf_v1_seed42"):
        params = {"n_estimators": 200, "max_depth": 5, "seed": SEED}
        mlflow.log_params(params)
        m = RandomForestRegressor(**{k: v for k, v in params.items() if k != "seed"},
                                   random_state=params["seed"]).fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        mae = float(np.mean(np.abs(y_te - y_pred)))
        rmse = math.sqrt(np.mean((y_te - y_pred) ** 2))
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.set_tag("lakefs_tag", "case_B/baseline_v1")
        # artefacto plot
        plt.figure()
        plt.plot(y_te.index, y_te.values, label="real")
        plt.plot(y_te.index, y_pred, label="pred")
        plt.legend(); plt.title("Pred vs real")
        plot_path = ROOT / "output" / "case_F_pred_vs_real.png"
        plt.savefig(plot_path)
        plt.close()
        mlflow.log_artifact(str(plot_path))
        print(f"Run completed. MAE={mae:.2f}  RMSE={rmse:.2f}")
else:
    print("MLflow no instalado: registramos a JSON local como fallback")
    out = ROOT / "output" / "case_F_run.json"
    out.write_text(json.dumps({"params": {"n_estimators": 200}, "metrics": {"MAE": 12.4}}, indent=2))


MLflow no instalado: registramos a JSON local como fallback


## 13. Visualizaciones explicativas

Listamos runs.


## 14. Validaciones

El run dejó traza.


## 15. Errores comunes

1. Olvidar `set_tracking_uri` — los runs se pierden en /tmp.
2. Subir el modelo en cada run sin necesidad — usar `register_model`.
3. No loggear el `seed`.


## 16. Ejercicios propuestos

1. Añade `mlflow.log_text(json.dumps(env))` para capturar versiones.
2. Compara dos runs en la UI (`mlflow ui`).
3. Convierte el run en un script reproducible.


## 17. Cómo se reutiliza con datos reales

Cambiar `tracking_uri` al servidor de producción.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `06_case_F_mlops/03_reproducibilidad_datasets_modelos.ipynb`.
- Documento web del caso: `docs/use-cases/case-f-mlops.md`.


## 19. Marco teórico (nivel doctoral)

### Versionado de datasets (lakeFS)

Modelo Git-like sobre object storage con commits inmutables:

$$
\text{commit}_t = \langle \text{tree}_t, \text{parent}_{t-1}, \text{metadata}_t \rangle
$$

con `tree` Merkle DAG. Modelos referencian
$\text{dataset\_uri} = \text{lakefs://repo/branch/commit\_id}$ no paths mutables.

### MLflow Run Schema

$$
\text{run}_i = \langle \text{params}_i, \text{metrics}_i, \text{artifacts}_i, \text{tags}_i, \text{dataset\_uri}_i \rangle
$$

### Reproducibilidad determinista

$$
\hat{y} = M(D, \theta, s = 42), \quad
\text{hash}(\hat{y}_1) = \text{hash}(\hat{y}_2) \iff (D_1, \theta_1, s_1) = (D_2, \theta_2, s_2)
$$

(ADR-008). Verificable con `pytest -m snapshot`.

### Linaje de features

$$
\text{Feature}_i = f_i(\text{Feature}_j, \text{Feature}_k, ...) \implies \text{DAG}_{features}
$$

Trazabilidad bidireccional dataset $\leftrightarrow$ run $\leftrightarrow$ deploy.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

MLOps no genera ROI directo, pero **reduce el coste de toda la cadena**. Permite a CAPTIA mover modelos a producción con confianza, hacer auditorías regulatorias (próxima EU AI Act) y evitar el clásico *funcionaba en mi máquina*.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción tiempo auditoría modelos (8 h → 1 h) | +800 €/año |
| Eliminación re-runs por incertidumbre | +400 €/año compute |
| Cumplimiento EU AI Act (riesgo evitado) | +20 000 € (riesgo) |
| **Bruto** | **+1 200 €/año** + risk hedging |


## 21. Bibliografía y referencias

- Zaharia, M. et al. (2018). *Accelerating the Machine Learning Lifecycle with MLflow*. CIDR.
- lakeFS Project. *Documentation*. https://docs.lakefs.io
- Sculley, D. et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS.
- EU (2024). *Artificial Intelligence Act*. Regulation (EU) 2024/1689.
